# Final Project — Trusted Retail Sales Lakehouse

**Domain:** E-commerce  |  **Data:** UCI Online Retail (541,909 rows, Dec-2010 → Dec-2011)

**Problem:** the raw sales export contains cancellations, missing customers, zero prices, duplicates and non-product codes → revenue reports built on it are wrong.

**Output:** a trusted Delta table + business analytics (KPIs, monthly revenue, top products, countries, customer segments).

```text
data.csv → Ingestion (batch + streaming) → Spark → Data Quality → Quality Gate
                                                                ↙        ↘
                                                             FAIL        PASS
                                                              ↓            ↓
                                                         Quarantine   Delta Lake → Analytics
```

# Part 1 — Libraries

In [ ]:
!pip -q install delta-spark loguru pyarrow

In [ ]:
import os, sys, json, shutil
from datetime import datetime
from loguru import logger

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

import matplotlib.pyplot as plt

# Part 2 — Logging

In [ ]:
LOG_FORMAT = "{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}"

logger.remove()                                         # avoid duplicate logs on re-run
logger.add(sys.stderr, format=LOG_FORMAT, level="INFO")
logger.add("pipeline_execution.log", format=LOG_FORMAT, level="INFO")

# Part 3 — Spark + Delta Lake

```text
landing_zone/        raw file
quarantine_zone/     failed batches
lakehouse/trusted_sales   stream_orders   gold/
```

In [ ]:
builder = (
    SparkSession.builder
    .appName("TrustedRetailLakehouse")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "4g")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

LANDING_ZONE    = "landing_zone"
QUARANTINE_ZONE = "quarantine_zone"
TRUSTED_PATH    = "lakehouse/trusted_sales"
STREAM_PATH     = "lakehouse/stream_orders"
GOLD_PATH       = "lakehouse/gold"

# start clean so the notebook can be re-run
for folder in [LANDING_ZONE, QUARANTINE_ZONE, "lakehouse"]:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder)

print("🚀 Spark", spark.version, "+ Delta Lake ready")

# Part 4 — Batch Ingestion

`data.csv → landing_zone/ → Spark DataFrame` (explicit schema, file encoding is ISO-8859-1 because of `£`).

In [ ]:
SOURCE_FILE = "data.csv"

if not os.path.exists(SOURCE_FILE):          # Colab: upload data.csv
    from google.colab import files
    SOURCE_FILE = list(files.upload().keys())[0]

raw_data_file = f"{LANDING_ZONE}/online_retail_batch.csv"
shutil.copy(SOURCE_FILE, raw_data_file)
logger.info(f"New raw data batch landed: {raw_data_file}")

In [ ]:
raw_schema = StructType([
    StructField("InvoiceNo",   StringType(),  True),   # "C..." = cancellation
    StructField("StockCode",   StringType(),  True),
    StructField("Description", StringType(),  True),
    StructField("Quantity",    IntegerType(), True),
    StructField("InvoiceDate", StringType(),  True),   # "12/1/2010 8:26", parsed later
    StructField("UnitPrice",   DoubleType(),  True),
    StructField("CustomerID",  StringType(),  True),
    StructField("Country",     StringType(),  True)
])

raw_df = (spark.read
          .option("header", True)
          .option("encoding", "ISO-8859-1")
          .schema(raw_schema)
          .csv(raw_data_file)
          .cache())

print("Raw rows:", raw_df.count())
raw_df.show(5, truncate=False)

In [ ]:
# A few real problems in the raw data
raw_df.filter(F.col("InvoiceNo").startswith("C")).show(2)               # cancellation
raw_df.filter(F.col("Description").isNull()).show(2)                    # missing values, price 0
raw_df.filter(F.col("StockCode").isin("POST", "M")).show(2)             # non-product codes

# Part 5 — Streaming Ingestion (Day 2 simulation)

Replay 3 real invoices as live events: `Producer → Broker → Processor → Delta`.

Processor rule: customer present, quantity > 0, price > 0, not a cancellation.

In [ ]:
# 536365 = normal order, C536379 = cancellation, 536414 = no customer + price 0
stream_source = raw_df.filter(F.col("InvoiceNo").isin("536365", "C536379", "536414")).collect()

def create_event(row):                                    # Producer
    return {
        "invoice_no": row["InvoiceNo"], "stock_code": row["StockCode"],
        "description": row["Description"], "quantity": row["Quantity"],
        "unit_price": row["UnitPrice"], "customer_id": row["CustomerID"],
        "country": row["Country"], "event_time": datetime.now().isoformat()
    }

def process_event(event):                                 # Processor
    reasons = []
    if event["customer_id"] is None:                   reasons.append("missing customer")
    if (event["quantity"] or 0) <= 0:                  reasons.append("quantity <= 0")
    if (event["unit_price"] or 0) <= 0:                reasons.append("price <= 0")
    if event["invoice_no"].startswith("C"):            reasons.append("cancellation")
    event["status"] = "VALID" if not reasons else "REJECTED"
    event["reason"] = ", ".join(reasons)
    return event

stream_schema = StructType([
    StructField("invoice_no",  StringType(),  True),
    StructField("stock_code",  StringType(),  True),
    StructField("description", StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("customer_id", StringType(),  True),
    StructField("country",     StringType(),  True),
    StructField("event_time",  StringType(),  True)
])

In [ ]:
broker = []            # Kafka in production
rejected_events = []

for row in stream_source:
    broker.append(create_event(row))                      # Producer → Broker
    event = process_event(broker.pop(0))                  # Broker → Processor

    if event["status"] == "VALID":                        # Processor → Sink
        record = [tuple(event[f.name] for f in stream_schema.fields)]
        spark.createDataFrame(record, stream_schema).write.format("delta").mode("append").save(STREAM_PATH)
        print(f"✅ {event['invoice_no']} {event['stock_code']:<7} → Delta")
    else:
        rejected_events.append(event)
        print(f"⛔ {event['invoice_no']} {event['stock_code']:<7} → REJECTED ({event['reason']})")

with open(f"{QUARANTINE_ZONE}/rejected_stream_events.json", "w") as f:
    json.dump(rejected_events, f, indent=4)

In [ ]:
stream_df = spark.read.format("delta").load(STREAM_PATH)
stream_df.select("invoice_no", "stock_code", "description", "quantity", "unit_price").show(truncate=False)

# Part 6 — DataQualityEngine (Spark)

Each row is checked against the rules. Failed rule names are stored in a `dq_errors` column.

| Dimension | Rule (row fails if) |
|---|---|
| Completeness | `CustomerID` or `Description` is NULL |
| Accuracy | `Quantity <= 0`, `UnitPrice <= 0`, or the sale was cancelled later |
| Uniqueness | the row is a duplicate of a previous row |
| Validity | `InvoiceNo` not `^\d{6}$`, `StockCode` not `^\d{5}[A-Za-z]*$` |

In [ ]:
from pyspark.sql import Window

class DataQualityEngine:

    def __init__(self, dataframe):
        self.df = dataframe
        self.results = {
            "timestamp": datetime.now().isoformat(),
            "metrics": {},
            "passed_all_gates": True
        }

    def run_all_checks(self):
        logger.info("Initiating automated data quality scan...")
        df = self.df

        # helper data for the "cancelled later" rule: C-invoices with the opposite quantity
        cancelled = (df.filter(F.col("InvoiceNo").startswith("C"))
                       .select("CustomerID", "StockCode", (-F.col("Quantity")).alias("Quantity"))
                       .dropDuplicates()
                       .withColumn("was_cancelled", F.lit(True)))
        df = df.join(cancelled, ["CustomerID", "StockCode", "Quantity"], "left")

        # helper for uniqueness: number each copy of an identical row (1 = first copy)
        dup_window = Window.partitionBy(*self.df.columns).orderBy(F.lit(1))
        df = df.withColumn("copy_no", F.row_number().over(dup_window))

        rules = {
            # 1. COMPLETENESS
            "completeness_customer_id":     F.col("CustomerID").isNull(),
            "completeness_description":     F.col("Description").isNull(),
            # 2. ACCURACY
            "accuracy_positive_quantity":   F.col("Quantity") <= 0,
            "accuracy_positive_unit_price": F.col("UnitPrice") <= 0,
            "accuracy_not_cancelled_later": F.col("was_cancelled").isNotNull(),
            # 3. UNIQUENESS
            "uniqueness_duplicate_row":     F.col("copy_no") > 1,
            # 4. VALIDITY
            "validity_invoice_no_format":   ~F.col("InvoiceNo").rlike(r"^\d{6}$"),
            "validity_stock_code_format":   ~F.col("StockCode").rlike(r"^\d{5}[A-Za-z]*$"),
        }

        # dq_errors = list of the rules this row failed (empty list = clean row)
        errors = F.array(*[F.when(cond, F.lit(name)) for name, cond in rules.items()])
        self.flagged_df = (df.withColumn("dq_errors", F.filter(errors, lambda e: e.isNotNull()))
                             .select(*self.df.columns, "dq_errors")   # keep original column order
                             .cache())

        # count failures per rule in ONE pass
        counts = self.flagged_df.agg(
            F.count("*").alias("total_rows"),
            F.sum((F.size("dq_errors") == 0).cast("int")).alias("passed_rows"),
            *[F.sum(F.array_contains("dq_errors", name).cast("int")).alias(name) for name in rules]
        ).first()

        for name in rules:
            self.results["metrics"][name] = {
                "value": int(counts[name]),
                "status": "PASSED" if counts[name] == 0 else "FAILED"
            }

        self.results["total_rows"]  = int(counts["total_rows"])
        self.results["passed_rows"] = int(counts["passed_rows"])
        self.results["failed_rows"] = int(counts["total_rows"] - counts["passed_rows"])
        self.results["passed_all_gates"] = self.results["failed_rows"] == 0
        return self.results

# Part 7 — Run the Quality Engine

In [ ]:
dq_engine = DataQualityEngine(raw_df)
dq_report = dq_engine.run_all_checks()
print(json.dumps(dq_report, indent=4))

# Part 8 — Transformation (for rows that pass)

Raw → trusted: trim text, upper-case codes, parse the date, add `InvoiceMonth` and `Revenue`.

In [ ]:
def to_trusted(df):
    return (df
        .withColumn("Description", F.trim("Description"))
        .withColumn("StockCode", F.upper("StockCode"))
        .withColumn("InvoiceTimestamp", F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
        .withColumn("InvoiceMonth", F.date_format("InvoiceTimestamp", "yyyy-MM"))
        .withColumn("Revenue", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
        .drop("InvoiceDate"))

# Part 9 — Quality Gate (split)

```text
               flagged rows
                    ↓
            dq_errors empty?
              ↙          ↘
           No             Yes
            ↓              ↓
       Quarantine     to_trusted() → Delta Lake
```

In [ ]:
def route_batch(flagged_df, dq_report, batch_name):

    passed_df = flagged_df.filter(F.size("dq_errors") == 0).drop("dq_errors")
    failed_df = flagged_df.filter(F.size("dq_errors") > 0) \
                          .withColumn("dq_errors", F.concat_ws(", ", "dq_errors"))

    # FAIL → Quarantine (data + report)
    quarantine_path = f"{QUARANTINE_ZONE}/{batch_name}_{int(datetime.now().timestamp())}"
    failed_df.write.mode("overwrite").parquet(quarantine_path)
    with open(f"{quarantine_path}_report.json", "w") as f:
        json.dump(dq_report, f, indent=4)
    logger.warning(f"{dq_report['failed_rows']} rows isolated to {quarantine_path}")

    # PASS → Delta Lake
    to_trusted(passed_df).write.format("delta").mode("overwrite").save(TRUSTED_PATH)
    logger.success(f"{dq_report['passed_rows']} rows passed → {TRUSTED_PATH}")

    return quarantine_path


quarantine_path = route_batch(dq_engine.flagged_df, dq_report, "online_retail")

### Investigate the quarantine: why did rows fail?

In [ ]:
quarantine_df = spark.read.parquet(quarantine_path)

quarantine_df.groupBy("dq_errors").count().orderBy(F.desc("count")).show(10, truncate=False)
quarantine_df.select("InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID", "dq_errors").show(5, truncate=False)

# Part 10 — Trusted Delta Table

In [ ]:
trusted_df = spark.read.format("delta").load(TRUSTED_PATH)
print("Trusted rows:", trusted_df.count())
trusted_df.show(5, truncate=False)

DeltaTable.forPath(spark, TRUSTED_PATH).history().select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
# Schema enforcement: raw data cannot be appended to the trusted table
try:
    raw_df.write.format("delta").mode("append").save(TRUSTED_PATH)
    print("⚠️ Write succeeded — investigate why.")
except Exception as e:
    print("🛡️ Write blocked:", str(e).splitlines()[0])

# Part 11 — Analytics (trusted data only)

In [ ]:
trusted_df.createOrReplaceTempView("sales")

BAR_COLOR = "#2a78d6"

def style_chart(ax, title, xlabel="", ylabel=""):
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_axisbelow(True)

## 11.1 KPIs

In [ ]:
spark.sql('''
    SELECT ROUND(SUM(Revenue), 0)                             AS total_revenue,
           COUNT(DISTINCT InvoiceNo)                          AS orders,
           COUNT(DISTINCT CustomerID)                         AS customers,
           COUNT(DISTINCT StockCode)                          AS products,
           ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS avg_order_value
    FROM sales
''').show()

## 11.2 Monthly revenue

Dec-2011 only has data until 9 Dec.

In [ ]:
monthly_revenue = spark.sql('''
    SELECT InvoiceMonth,
           ROUND(SUM(Revenue), 0)     AS revenue,
           COUNT(DISTINCT InvoiceNo)  AS orders,
           COUNT(DISTINCT CustomerID) AS customers
    FROM sales
    GROUP BY InvoiceMonth
    ORDER BY InvoiceMonth
''')
monthly_revenue.show(13)

pdf = monthly_revenue.toPandas()
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(pdf["InvoiceMonth"], pdf["revenue"] / 1000, color=BAR_COLOR, width=0.6)
ax.grid(axis="y", alpha=0.25)
style_chart(ax, "Monthly revenue", ylabel="£ thousands")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 11.3 Top 10 products

In [ ]:
top_products = spark.sql('''
    SELECT StockCode,
           FIRST(Description)     AS product,
           SUM(Quantity)          AS units_sold,
           ROUND(SUM(Revenue), 0) AS revenue
    FROM sales
    GROUP BY StockCode
    ORDER BY revenue DESC
    LIMIT 10
''')
top_products.show(truncate=False)

pdf = top_products.toPandas().iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(pdf["product"], pdf["revenue"] / 1000, color=BAR_COLOR, height=0.6)
ax.grid(axis="x", alpha=0.25)
style_chart(ax, "Top 10 products by revenue", xlabel="£ thousands")
plt.tight_layout(); plt.show()

## 11.4 Revenue by country

In [ ]:
country_revenue = spark.sql('''
    SELECT Country,
           COUNT(DISTINCT CustomerID)                               AS customers,
           ROUND(SUM(Revenue), 0)                                   AS revenue,
           ROUND(SUM(Revenue) * 100 / SUM(SUM(Revenue)) OVER (), 1) AS share_pct
    FROM sales
    GROUP BY Country
    ORDER BY revenue DESC
    LIMIT 10
''')
country_revenue.show(truncate=False)

## 11.5 Customer segments (RFM)

**R**ecency (days since last order), **F**requency (orders), **M**onetary (revenue).

```text
Champions  → last 30 days AND 10+ orders
Loyal      → 5+ orders
New/Recent → last 60 days
At Risk    → no order for 180+ days
Regular    → everyone else
```

In [ ]:
rfm = spark.sql('''
    WITH ref AS (SELECT DATE_ADD(MAX(TO_DATE(InvoiceTimestamp)), 1) AS ref_date FROM sales)
    SELECT CustomerID,
           DATEDIFF((SELECT ref_date FROM ref), MAX(TO_DATE(InvoiceTimestamp))) AS recency_days,
           COUNT(DISTINCT InvoiceNo)                                             AS frequency,
           ROUND(SUM(Revenue), 2)                                                AS monetary
    FROM sales
    GROUP BY CustomerID
''')

customer_segments = rfm.withColumn(
    "segment",
    F.when((F.col("recency_days") <= 30) & (F.col("frequency") >= 10), "Champions")
     .when(F.col("frequency") >= 5,      "Loyal")
     .when(F.col("recency_days") <= 60,  "New/Recent")
     .when(F.col("recency_days") > 180,  "At Risk")
     .otherwise("Regular")
)
customer_segments.createOrReplaceTempView("customer_segments")

segment_summary = spark.sql('''
    SELECT segment,
           COUNT(*)                                                  AS customers,
           ROUND(SUM(monetary), 0)                                   AS revenue,
           ROUND(SUM(monetary) * 100 / SUM(SUM(monetary)) OVER (), 1) AS share_pct,
           ROUND(AVG(frequency), 1)                                  AS avg_orders
    FROM customer_segments
    GROUP BY segment
    ORDER BY revenue DESC
''')
segment_summary.show(truncate=False)

pdf = segment_summary.toPandas()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(pdf["segment"], pdf["share_pct"], color=BAR_COLOR, width=0.55)
for i, v in enumerate(pdf["share_pct"]):
    ax.text(i, v + 1, f"{v}%", ha="center")
style_chart(ax, "Share of revenue by customer segment", ylabel="% of revenue")
plt.tight_layout(); plt.show()

## 11.6 Save results as Gold Delta tables

In [ ]:
gold_tables = {
    "monthly_revenue": monthly_revenue,
    "top_products": top_products,
    "country_revenue": country_revenue,
    "customer_segments": customer_segments,
}
for name, df in gold_tables.items():
    df.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/{name}")

logger.success(f"Gold tables saved: {list(gold_tables)}")

# Part 12 — Summary

In [ ]:
print(json.dumps({
    "raw_rows": dq_report["total_rows"],
    "passed_rows_to_delta": dq_report["passed_rows"],
    "failed_rows_to_quarantine": dq_report["failed_rows"],
    "stream_stored": stream_df.count(),
    "stream_rejected": len(rejected_events),
}, indent=4))

spark.stop()